# 🕵️‍♂️ Agentic Sherlock AI — Google Colab GPU Inference Bridge

**A Hierarchical Cognitive Multi-Agent Framework for Cyber Attack Reconstruction & Digital Forensics**  
Team Number: **G1266** | Mentor: **Para Upendar**  

This notebook provides a free, cloud-hosted **T4/A100 GPU Inference Microservice** for your local Sherlock AI desktop application (`sherf`).  
It loads **Qwen2.5-7B-Instruct** in 4-bit NF4 precision with the fine-tuned cybersecurity LoRA adapters, and exposes a high-throughput tunnel endpoint.

In [ ]:
# 1. Verify GPU Availability
!nvidia-smi

# 2. Install Required Dependencies
!pip install -q torch transformers peft bitsandbytes accelerate fastapi uvicorn pyngrok nest_asyncio pydantic

In [ ]:
import os
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

BASE_MODEL_ID = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit"
print(f"[1/3] Loading Tokenizer for {BASE_MODEL_ID}...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, trust_remote_code=True)

print(f"[2/3] Loading 4-bit Quantized Base Model onto GPU...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)
model.eval()


In [ ]:
# 4. Launch Cloudflare Tunnel and Start Service
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

import subprocess, time, threading, re

def start_server():
    uvicorn.run(app, host="127.0.0.1", port=8000, log_level="warning")

server_thread = threading.Thread(target=start_server, daemon=True)
server_thread.start()
time.sleep(3)

print("Starting Cloudflare tunnel bridge...")
tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True
)

tunnel_url = None
for line in tunnel_proc.stderr:
    match = re.search(r"https://[-a-zA-Z0-9]+\.trycloudflare\.com", line)
    if match:
        tunnel_url = match.group(0)
        print("=" * 68)
        print(f"COLAB GPU INFERENCE TUNNEL ACTIVE: {tunnel_url}")
        print("=" * 68)
        print("Paste this URL into your Sherlock AI settings or environment variable:")
        print(f'COLAB_GPU_INFERENCE_URL = "{tunnel_url}"')
        print("=" * 68)
        print("Service is LIVE and serving neural inferences on GPU. Keep this cell running!")
        break

try:
    while True:
        time.sleep(1)
except KeyboardInterrupt:
    print("\nStopping tunnel...")
    tunnel_proc.terminate()


In [ ]:
# 4. Launch Cloudflare Tunnel & Start Service
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

import subprocess, time, threading

def start_server():
    uvicorn.run(app, host="127.0.0.1", port=8000)

threading.Thread(target=start_server, daemon=True).start()
time.sleep(2)

print("Starting Cloudflare tunnel...")
tunnel_proc = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"], stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)

for line in tunnel_proc.stderr:
    if "trycloudflare.com" in line:
        for word in line.split():
            if "trycloudflare.com" in word:
                print("=" * 65)
                print(f"🎉 COLAB GPU INFERENCE TUNNEL ACTIVE: {word}")
                print("Paste this URL into your Sherlock AI settings or environment variable:")
                print(f'COLAB_GPU_INFERENCE_URL = "{word}"')
                print("=" * 65)
                break
        break